In [37]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
import json


In [38]:
# Generate a synthetic dataset with 50 input features
# Let's create 1000 samples with 50 features each and binary labels
np.random.seed(42)  # For reproducibility

X = np.random.rand(1000, 50)  # 1000 samples, 50 features
y = np.random.randint(2, size=(1000, 1))  # Binary labels (0 or 1)

print(X.shape, y.shape)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# print(y_train.shape, y_test.shape)
# print(X_train.shape, X_test.shape)


(1000, 50) (1000, 1)


In [39]:
# Define a single-layer perceptron model
model = Sequential([
    Dense(1, input_dim=50, activation='relu'),  # One neuron, 50 inputs, sigmoid activation
    # BatchNormalization()
])

# Compile the model
model.compile(optimizer='sgd',  # Stochastic Gradient Descent
              loss='binary_crossentropy',  # Loss function for binary classification
              metrics=['accuracy'])


# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=10, verbose=1)


Epoch 1/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 698us/step - accuracy: 0.5115 - loss: 4.8007
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 523us/step - accuracy: 0.5318 - loss: 7.4646
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 454us/step - accuracy: 0.5158 - loss: 7.7194
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 292us/step - accuracy: 0.5136 - loss: 7.7543
Epoch 5/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 284us/step - accuracy: 0.5274 - loss: 7.5350
Epoch 6/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 289us/step - accuracy: 0.5463 - loss: 7.2332
Epoch 7/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 274us/step - accuracy: 0.5001 - loss: 7.9698
Epoch 8/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 287us/step - accuracy: 0.5413 - loss: 7.3135
Epoch 9/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 275us/step - accuracy: 0.5100 - loss: 7.8122
Epoch 10/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 269us/step - accuracy: 0.4978 - loss: 8.0070
Epoch 11/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 280us/step - accuracy: 0.5063 - loss: 7.8704
Epoch 12/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 290us/step

In [40]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4660 - loss: 8.5134 
Test Loss: 8.1306
Test Accuracy: 0.4900


In [41]:
# Access the weights
weights, bias = model.get_weights()

# Save the weights to a JSON file
weights_dict = {
    "weights": weights.tolist(),
    "bias": bias.tolist()
}

# print(weights)

In [42]:
with open("perceptron_weights.json", "w") as fw:
    json.dump(weights_dict, fw)

print("Weights and bias saved to perceptron_weights.json")

Weights and bias saved to perceptron_weights.json


In [43]:
# Read the JSON file
with open('perceptron_weights.json', 'r') as fr:
    data = json.load(fr)

# Convert the JSON data to a format suitable for Verilog
with open('weights_values.mem', 'w') as fmem:
    for weight_value in data['weights']:
        weight_valueQ15 = weight_value[0] * 2**15 # Convert to Q15.0 fixed-point format
        weight_valueQ15 = int(weight_valueQ15) # Convert to integer
        # Get the raw binary representation
        binary_representation = bin(weight_valueQ15 & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")
        # print(weight_valueQ15)    
        
    bias_value = data['bias']
    bias_valueQ15 = bias_value[0] * 2**15 # Convert to Q15.0 fixed-point format
    bias_valueQ15 = int(bias_valueQ15) # Convert to integer
    # Get the raw binary representation
    binary_representation = bin(bias_valueQ15 & 0xFFFF)[2:].zfill(16)
    fmem.write(f"{binary_representation}\n") 
    

# Save iput data to a file
XQ15 = X * 2**15
XQ15 = XQ15.astype(np.int16)

with open('input_values.mem', 'w') as fmem:
    for value in XQ15[0]:
        binary_representation = bin(value & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")  # Convert value to float before formatting as binary
# print(XQ15)

In [44]:
# Example single input (make sure it has the correct shape)
single_input = X[0].reshape(1, -1)

# Print the input value
# print("Input value for the single input:", single_input)

# Make a prediction
prediction = model.predict(single_input)

predictionQ15 = prediction[0][0] * 2**15 # Convert to Q15.0 fixed-point format

# Print the prediction and the classified class
print("Prediction for the single input:", prediction)
print("Prediction for the single input in Q15.0 format:", int(predictionQ15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction for the single input: [[2.0298643]]
Prediction for the single input in Q15.0 format: 66514


In [45]:
# Assuming single_input and weights are already defined as numpy arrays
# Slice the first ten elements
single_input_ = single_input[0]
weights_ = weights[0:50]

print(single_input_)
print(weights_)

# Perform the dot product with the first ten elements
product = np.dot(single_input_, weights_)
print("Product:", product)
print(product +bias_value[0])

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452
 0.05808361 0.86617615 0.60111501 0.70807258 0.02058449 0.96990985
 0.83244264 0.21233911 0.18182497 0.18340451 0.30424224 0.52475643
 0.43194502 0.29122914 0.61185289 0.13949386 0.29214465 0.36636184
 0.45606998 0.78517596 0.19967378 0.51423444 0.59241457 0.04645041
 0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735
 0.30461377 0.09767211 0.68423303 0.44015249 0.12203823 0.49517691
 0.03438852 0.9093204  0.25877998 0.66252228 0.31171108 0.52006802
 0.54671028 0.18485446]
[[ 0.34535104]
 [ 0.2278963 ]
 [ 0.1707002 ]
 [ 0.14195268]
 [ 0.21648645]
 [-0.15221444]
 [ 0.10777151]
 [-0.20224676]
 [-0.07069974]
 [-0.05351491]
 [-0.07502781]
 [-0.05461428]
 [ 0.00182127]
 [ 0.06601355]
 [-0.13589965]
 [-0.07363608]
 [-0.08344103]
 [ 0.5050587 ]
 [-0.15392077]
 [ 0.24296507]
 [ 0.3700724 ]
 [ 0.2695264 ]
 [ 0.1675413 ]
 [ 0.06296226]
 [ 0.04130093]
 [ 0.0629135 ]
 [ 0.2353347 ]
 [ 0.2529351 ]
 [-0.21272191]
 [ 0.